# Notebook — Kernel PCA: the kernel trick made visible

Two interlocking "moons" are **not** linearly separable. We show:
- **Linear PCA** just rotates the data — the classes stay tangled, and the first
  component does not separate them.
- **Kernel PCA** (RBF kernel, via the centred **Gram matrix**) maps the data into
  a feature space where the classes fall cleanly apart in the top two components.

All from kernel evaluations — we never build the (infinite-dimensional) feature
map. NumPy + Matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (10, 4)

def make_moons(n=300, noise=0.08):
    n1, n2 = n // 2, n - n // 2
    t1 = np.linspace(0, np.pi, n1); m1 = np.stack([np.cos(t1), np.sin(t1)], 1)
    t2 = np.linspace(0, np.pi, n2); m2 = np.stack([1 - np.cos(t2), 0.5 - np.sin(t2)], 1)
    X = np.vstack([m1, m2]) + noise * rng.standard_normal((n, 2))
    y = np.concatenate([np.zeros(n1), np.ones(n2)]).astype(int)
    return X, y

X, y = make_moons()
plt.figure(figsize=(5, 5))
plt.scatter(X[y==0, 0], X[y==0, 1], s=12, c='tab:blue', label='class 0')
plt.scatter(X[y==1, 0], X[y==1, 1], s=12, c='tab:red',  label='class 1')
plt.gca().set_aspect('equal'); plt.legend(); plt.title('two moons — not linearly separable'); plt.show()

## Linear PCA — and why it can't help here

PCA on 2D data is just a rotation onto the directions of maximum variance. The
classes stay interleaved, and projecting onto the first principal component
leaves the two histograms overlapping.

In [ ]:
Xc = X - X.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
pc = Xc @ Vt.T                      # PCA scores (N, 2)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(pc[y==0, 0], pc[y==0, 1], s=12, c='tab:blue')
ax[0].scatter(pc[y==1, 0], pc[y==1, 1], s=12, c='tab:red')
ax[0].set_xlabel('PC1'); ax[0].set_ylabel('PC2'); ax[0].set_title('linear PCA — still tangled'); ax[0].set_aspect('equal')
ax[1].hist(pc[y==0, 0], bins=30, alpha=0.6, color='tab:blue', label='class 0')
ax[1].hist(pc[y==1, 0], bins=30, alpha=0.6, color='tab:red',  label='class 1')
ax[1].set_xlabel('projection onto PC1'); ax[1].legend(); ax[1].set_title('PC1 does not separate the classes')
plt.tight_layout(); plt.show()

## Kernel PCA (RBF)

Build the RBF kernel matrix \(K_{ij}=\exp(-\gamma\lVert x_i-x_j\rVert^2)\),
**centre it in feature space**, and eigen-decompose. The top eigenvectors are the
kernel-PCA components — computed entirely from kernel values, never from the
(infinite) feature map. The classes now separate.

In [ ]:
def rbf_gram(X, gamma):
    sq = np.sum(X**2, axis=1)
    D2 = np.maximum(sq[:, None] + sq[None, :] - 2 * X @ X.T, 0.0)   # squared distances
    return np.exp(-gamma * D2)

gamma = 8.0
K = rbf_gram(X, gamma)
n = K.shape[0]
O = np.ones((n, n)) / n
Kc = K - O @ K - K @ O + O @ K @ O          # centre in feature space

evals, evecs = np.linalg.eigh(Kc)            # ascending
order = np.argsort(evals)[::-1]
evals, evecs = evals[order], evecs[:, order]
kpca = evecs[:, :2]                           # top-2 kernel-PCA components (per training point)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(kpca[y==0, 0], kpca[y==0, 1], s=12, c='tab:blue')
ax[0].scatter(kpca[y==1, 0], kpca[y==1, 1], s=12, c='tab:red')
ax[0].set_xlabel('kPCA 1'); ax[0].set_ylabel('kPCA 2'); ax[0].set_title('kernel PCA — classes separate')
ax[1].hist(kpca[y==0, 0], bins=30, alpha=0.6, color='tab:blue', label='class 0')
ax[1].hist(kpca[y==1, 0], bins=30, alpha=0.6, color='tab:red',  label='class 1')
ax[1].set_xlabel('projection onto kPCA 1'); ax[1].legend(); ax[1].set_title('kPCA 1 separates the classes')
plt.tight_layout(); plt.show()

## Exercises

1. Sweep the RBF width `gamma` (0.5, 2, 8, 50). Too small ≈ linear PCA; too large
   overfits to individual points. Find the range that separates the moons.
2. Swap the moons for two **concentric circles** and repeat — kernel PCA should
   still separate them.
3. Standardise the features before the kernel; does it change the result? (It
   matters when dimensions have different scales.)
4. Use the kPCA-1 coordinate as a 1-D feature for a simple threshold classifier
   and measure accuracy vs the same with PC1.
5. Connect to shape analysis: run kernel PCA on the aligned shape vectors from
   `01_shape_analysis_procrustes_pca.ipynb` and compare modes to linear PCA.

**Context:** this is the nonlinear upgrade of the PCA you used for shape models,
and the kernel trick behind kernel k-means / GMM segmentation. It rounds out the
Week-4 classical toolkit.